# SQL JOIN 多表连接（习题）

In [34]:
import duckdb

%load_ext sql
%sql duckdb:///:memory:

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [35]:
%%sql
CREATE OR REPLACE VIEW sales AS SELECT * FROM '../data/sales.csv';
SELECT COUNT(*) FROM sales;

Running query in 'duckdb:///:memory:'

count_star()
500


In [36]:
%%sql
-- 客户信息表:故意让它和 sales 里的 customer_id "不完全对齐"
CREATE OR REPLACE TABLE customers AS
SELECT * FROM (VALUES
    ('C001', 'Alice',   'VIP'),
    ('C002', 'Bob',     'Normal'),
    ('C003', 'Charlie', 'VIP'),
    ('C004', 'Diana',   'Normal'),
    ('C999', 'Eve',     'VIP')      -- C999 在 sales 里不存在(故意的)
) AS t(customer_id, customer_name, level);

SELECT * FROM customers;

Running query in 'duckdb:///:memory:'

customer_id,customer_name,level
C001,Alice,VIP
C002,Bob,Normal
C003,Charlie,VIP
C004,Diana,Normal
C999,Eve,VIP


In [37]:
%%sql
CREATE OR REPLACE TABLE countries AS
SELECT * FROM (VALUES
    ('US',      'North America'),
    ('Germany', 'Europe'),
    ('France',  'Europe'),
    ('Japan',   'Asia')
) AS t(country, region);

SELECT * FROM countries;

Running query in 'duckdb:///:memory:'

country,region
US,North America
Germany,Europe
France,Europe
Japan,Asia


Easy

1. 用 INNER JOIN 把 sales 和 customers 连起来,输出 order_id, customer_name, level, total。

In [38]:
%%sql
SELECT 
    s.order_id,
    c.customer_name,
    c.level,
    s.total
FROM sales s
INNER JOIN customers c ON s.customer_id = c.customer_id
LIMIT 20;

Running query in 'duckdb:///:memory:'

order_id,customer_name,level,total
O1001,Diana,Normal,99
O1004,Charlie,VIP,495
O1009,Bob,Normal,297
O1010,Charlie,VIP,1797
O1012,Charlie,VIP,897
O1013,Charlie,VIP,1299
O1016,Diana,Normal,2396
O1019,Charlie,VIP,897
O1022,Bob,Normal,2598
O1024,Diana,Normal,1198


2. 用 LEFT JOIN 连 sales 和 customers,输出 order_id, customer_id, customer_name。观察:customer_name 为 NULL 的行说明什么?

In [39]:
%%sql
SELECT 
    s.order_id,
    s.customer_id,
    c.customer_name
FROM sales s
LEFT JOIN customers c ON s.customer_id = c.customer_id
LIMIT 20;

Running query in 'duckdb:///:memory:'

order_id,customer_id,customer_name
O1001,C004,Diana
O1004,C003,Charlie
O1009,C002,Bob
O1010,C003,Charlie
O1012,C003,Charlie
O1013,C003,Charlie
O1016,C004,Diana
O1019,C003,Charlie
O1022,C002,Bob
O1024,C004,Diana


参考答案

题目要求你观察 customer_name 为 NULL 的行说明什么。你的 LIMIT 20 结果里恰好前 10 行都是 C002/C003/C004(都能匹配上),所以你没看到 NULL 行。这是个隐藏问题:你以为做完了,其实没观察到题目要的现象。
补一句验证——专门把 NULL 行捞出来看:

In [40]:
%%sql
SELECT s.order_id, s.customer_id, c.customer_name
FROM sales s
LEFT JOIN customers c ON s.customer_id = c.customer_id
WHERE c.customer_name IS NULL
LIMIT 10;

Running query in 'duckdb:///:memory:'

order_id,customer_id,customer_name
O1000,C007,None
O1002,C005,None
O1003,C007,None
O1005,C008,None
O1006,C005,None
O1007,C005,None
O1008,C007,None
O1011,C007,None
O1014,C008,None
O1015,C005,None


customer_name 为 NULL 说明:这笔订单的 customer_id 在 customers 表里查不到(比如 C005)。LEFT JOIN 保留了订单,但右表没东西填,只能填 NULL。

教训:LIMIT 20 看到的「前 20 行」不代表全部。题目让你观察某现象时,要主动构造能看到该现象的查询,别指望默认排序刚好给你。这条和你弱点 #19「自己造验证用例」是一回事。

3. 用 JOIN 把 sales 和 countries 连起来,输出每个订单的 order_id, country, region, total。先想:该用 INNER 还是 LEFT?(提示:sales 里的 country 会不会有 countries 表里没有的?自己查一下再决定。)

In [41]:
%%sql
SELECT 
    s.order_id,
    s.country,
    co.region,
    s.total
FROM sales s 
INNER JOIN countries co ON s.country = co.country
LIMIT 20;

Running query in 'duckdb:///:memory:'

order_id,country,region,total
O1000,Germany,Europe,2598
O1001,US,North America,99
O1002,US,North America,396
O1003,US,North America,396
O1004,France,Europe,495
O1009,Germany,Europe,297
O1014,France,Europe,2997
O1015,France,Europe,3996
O1016,US,North America,2396
O1017,US,North America,1999


参考答案

你用了 INNER JOIN。结果对不对取决于一个你没验证的前提:sales 里的 country 是不是都在 countries 表里?

看你自己题 9 的输出就露馅了——O1010 的 country 是 UK,而你的 countries 表只有 US/Germany/France/Japan,没有 UK!也就是说 sales 里存在 countries 表没有的国家。

那么题 3 用 INNER JOIN 的后果:所有 UK(以及其他未登记国家)的订单被悄悄丢掉了。题目明确提示「自己查一下再决定」,你跳过了这步。

正确做法是先查:

In [42]:
%%sql
-- sales 里有哪些 country 不在 countries 表?
SELECT DISTINCT s.country
FROM sales s
LEFT JOIN countries co ON s.country = co.country
WHERE co.country IS NULL;

Running query in 'duckdb:///:memory:'

country
UK
China


查完发现有 UK 等,就该用 LEFT JOIN(否则丢单)。这题不算你错——但「跳过验证、直接选 INNER」这个动作是 bug 思维。JOIN 之前永远先确认两表的 key 对齐情况。

Medium

4. 找出「有订单、但不在 customers 表里」的所有 customer_id(去重)。用 LEFT JOIN + IS NULL。

In [43]:
%%sql
SELECT 
    DISTINCT s.customer_id
FROM sales s 
LEFT JOIN customers c ON s.customer_id = c.customer_id
WHERE c.customer_id IS NULL
ORDER BY s.customer_id
LIMIT 20;

Running query in 'duckdb:///:memory:'

customer_id
C005
C006
C007
C008


5. 同样的需求(题 4),改用 Day 7 学的 NOT EXISTS 再写一遍。对比两种写法的结果是否一致。

In [44]:
%%sql
SELECT
    DISTINCT s.customer_id
FROM sales s 
WHERE NOT EXISTS(
    SELECT 1
    FROM customers c 
    WHERE s.customer_id = c.customer_id
)
ORDER BY s.customer_id
LIMIT 20;

Running query in 'duckdb:///:memory:'

customer_id
C005
C006
C007
C008


6. 按 region(大区)统计销售总额:连 sales 和 countries,按 region 分组求 SUM(total),降序。
注意:sales 里 country 匹配不上 countries 的订单,你希望算进去还是排除?想清楚再决定 JOIN 类型。

In [45]:
%%sql
SELECT
    SUM(s.total) AS region_total,
    co.region
FROM sales s 
LEFT JOIN countries co ON s.country = co.country
GROUP BY co.region
ORDER BY region_total DESC

Running query in 'duckdb:///:memory:'

region_total,region
614918,None
352185,Europe
300613,North America


参考答案

这个 None(NULL)是哪来的?你用 LEFT JOIN,sales 里 country 匹配不上 countries 的订单(UK 等),co.region 填 NULL,GROUP BY co.region 就多出一个「NULL 组」。

问题不在于 bug 本身,而在于你没做题目要求的决策。 题目原话:「sales 里 country 匹配不上的订单,你希望算进去还是排除?想清楚再决定 JOIN 类型。」你直接用了 LEFT,既没排除、也没给那个 NULL 组一个像样的名字,留了个 None 在结果里——这是「没想清楚」的表现。

两种都对,但你要明确选一个并知道自己在选什么:

In [46]:
%%sql
-- 选择 A:只统计已登记国家(排除未匹配)→ 用 INNER JOIN
SELECT co.region, SUM(s.total) AS region_total
FROM sales s
INNER JOIN countries co ON s.country = co.country
GROUP BY co.region
ORDER BY region_total DESC;

Running query in 'duckdb:///:memory:'

region,region_total
Europe,352185
North America,300613


In [47]:
%%sql
-- 选择 B:未匹配的也要算,但归到"Unknown"组,不留 None
SELECT
    COALESCE(co.region, 'Unknown') AS region,
    SUM(s.total) AS region_total
FROM sales s
LEFT JOIN countries co ON s.country = co.country
GROUP BY COALESCE(co.region, 'Unknown')
ORDER BY region_total DESC;

Running query in 'duckdb:///:memory:'

region,region_total
Unknown,614918
Europe,352185
North America,300613


新函数 COALESCE(a, b):返回第一个非 NULL 的值。COALESCE(co.region, 'Unknown') = 「region 有值就用 region,是 NULL 就用 'Unknown'」。这是处理 JOIN 后 NULL 的标准手段,记进笔记。

Python ↔ SQL 对照:COALESCE(a, b) ≈ Python 的 a if a is not None else b,也 ≈ d.get(key, default)(你 Day 2 笔记里的「安全访问」)。同一个思想。

7. 按客户等级 level(VIP / Normal)统计:每个等级有多少笔订单、总销售额多少。连 sales 和 customers,按 level 分组。

In [48]:
%%sql
SELECT 
    COUNT(s.customer_id) AS order_count,
    SUM(s.total) AS total_sales,
    c.level
FROM sales s 
LEFT JOIN customers c ON s.customer_id = c.customer_id
GROUP BY c.level
ORDER BY total_sales DESC;

Running query in 'duckdb:///:memory:'

order_count,total_sales,level
243,638441,None
135,344213,VIP
122,285062,Normal


参考答案

结果里又出现了 None 组(243 笔订单)。原因一样:LEFT JOIN 后,customer_id 匹配不上 customers 的订单,level 是 NULL。

这题你没踩坑反而踩对了一半——题目要的就是「按 level 分组」,而那 243 笔「未登记客户」的订单确实没有 level。但还是建议用 COALESCE(c.level, '未登记') 把它显式命名,别留 None。

另外一个小点:COUNT(s.customer_id)。你数的是「customer_id 非 NULL 的行数」。这里 sales 的 customer_id 不会是 NULL,所以碰巧对。但统计「订单数」的标准写法是 COUNT(*)——数行数就用 COUNT(*),别用某个具体列(万一那列有 NULL 就少数了)。这是你 Day 5 学的 COUNT 三兄弟,别忘。

Hard

8. 找出「在 customers 表里登记了、但从来没下过任何订单」的客户(输出 customer_id, customer_name)。两种写法都写:(a) JOIN 版 (b) NOT EXISTS 版。

In [49]:
%%sql
SELECT 
    c.customer_id,
    c.customer_name
FROM customers c
LEFT JOIN sales s ON c.customer_id = s.customer_id
WHERE s.customer_id IS NULL;

Running query in 'duckdb:///:memory:'

customer_id,customer_name
C999,Eve


In [50]:
%%sql
SELECT 
    c.customer_id,
    c.customer_name
FROM customers c
WHERE NOT EXISTS (
    SELECT 1
    FROM sales s 
    WHERE s.customer_id = c.customer_id
);

Running query in 'duckdb:///:memory:'

customer_id,customer_name
C999,Eve


9. 三表链式 JOIN:输出每笔订单的 order_id, customer_name, level, region, total,三张表全连上。matching 不上的订单也要保留(想清楚用哪种 JOIN)。

In [51]:
%%sql
SELECT 
    s.order_id,
    c.customer_name,
    c.level,
    co.region,
    s.total
FROM sales s
LEFT JOIN customers c ON s.customer_id = c.customer_id
LEFT JOIN countries co ON s.country = co.country
LIMIT 20;

Running query in 'duckdb:///:memory:'

order_id,customer_name,level,region,total
O1001,Diana,Normal,North America,99
O1004,Charlie,VIP,Europe,495
O1009,Bob,Normal,Europe,297
O1016,Diana,Normal,North America,2396
O1022,Bob,Normal,Europe,2598
O1024,Diana,Normal,Europe,1198
O1029,Diana,Normal,Europe,598
O1031,Alice,VIP,Europe,198
O1032,Diana,Normal,North America,4995
O1037,Alice,VIP,Europe,5196


10. —(JOIN 膨胀陷阱,今日最重要)

自己造一个刁钻测试:新建一张 promo 表,故意让一个 customer_id 出现两次(模拟「一个客户参加了两个促销活动」):

In [52]:
%%sql
CREATE OR REPLACE TABLE promo AS
SELECT * FROM (VALUES
    ('C003', 'SpringSale'),
    ('C003', 'SummerSale'),     -- C003 出现两次!
    ('C004', 'SpringSale')
) AS t(customer_id, promo_name);

Running query in 'duckdb:///:memory:'

Count


然后:

(a) 把 sales LEFT JOIN 这张 promo 表,观察 C003 的订单行数有什么变化

(b) 在笔记里解释:为什么 C003 的每一笔订单都「变成了两行」?

(c) 思考:如果这时你对 join 后的结果做 SUM(total),金额会怎样?

In [53]:
%%sql
SELECT 
    s.*,
    p.promo_name
FROM sales s
LEFT JOIN promo p ON s.customer_id = p.customer_id
LIMIT 20;

Running query in 'duckdb:///:memory:'

order_id,customer_id,product,category,quantity,price,order_date,country,total,promo_name
O1001,C004,Keyboard,Accessory,1,99,2024-01-01,US,99,SpringSale
O1004,C003,Phone,Mobile,5,99,2024-01-03,France,495,SummerSale
O1010,C003,Monitor,Computer,3,599,2024-01-08,UK,1797,SummerSale
O1012,C003,Monitor,Computer,3,299,2024-01-09,UK,897,SummerSale
O1013,C003,Phone,Mobile,1,1299,2024-01-10,UK,1299,SummerSale
O1016,C004,Phone,Mobile,4,599,2024-01-12,US,2396,SpringSale
O1019,C003,Laptop,Computer,3,299,2024-01-14,UK,897,SummerSale
O1024,C004,Phone,Mobile,2,599,2024-01-18,France,1198,SpringSale
O1029,C004,Keyboard,Accessory,2,299,2024-01-22,Germany,598,SpringSale
O1032,C004,Laptop,Computer,5,999,2024-01-24,US,4995,SpringSale


参考答案

JOIN 膨胀(fan-out):LEFT JOIN 的规则是「左表每一行,去右表找所有匹配的行,有几行匹配就复制成几行」。promo 表里 C003 出现 2 次,所以 sales 里 C003 的每一笔订单都会和 2 条 promo 记录各配一次 → 复制成 2 行。

后果:如果对膨胀后的结果 SUM(s.total),C003 的每笔金额都被算了 2 次,总额虚高。C004 只匹配 1 条,不受影响。所以这不是「整体翻倍」,是「C003 部分翻倍」——更隐蔽,更难发现。

验证一下，让你亲眼看见金额虚高：

In [54]:
%%sql
-- C003 真实总额 vs JOIN 后总额
SELECT
    (SELECT SUM(total) FROM sales WHERE customer_id='C003') AS 真实总额,
    SUM(s.total) AS join后总额
FROM sales s
LEFT JOIN promo p ON s.customer_id = p.customer_id
WHERE s.customer_id = 'C003';

Running query in 'duckdb:///:memory:'

真实总额,join后总额
160212,320424


因为promo的表内有两个C003订单，根据这个聚合，就会导致每一笔订单变成两行。

金额会翻倍。